In [93]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score 
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
import pickle

In [94]:
loan_df = pd.read_csv("loandata.csv")

In [95]:
loan_df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [96]:
loan_df = loan_df.dropna()

In [97]:
loan_df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
5,LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban,Y


In [98]:
X = loan_df.drop(columns=["Loan_ID", "Loan_Status"])
y = loan_df["Loan_Status"].map({"Y": 1, "N": 0})

X["Gender"] = X["Gender"].map({"Male":1 , "Female":0})
X["Education"] = X["Education"].map({"Graduate":1 , "Not Graduate":0})
X["Self_Employed"] = X["Self_Employed"].map({"Yes":1 , "No":0})
X["Married"] = X["Married"].map({"Yes":1 , "No":0})

X = pd.get_dummies(
    X,
    columns=["Property_Area"],
    dtype=int
)

X["Dependents"] = X["Dependents"].replace("3+", "3")
X["Dependents"] = pd.to_numeric(X["Dependents"])

X = pd.get_dummies(
    X,
    drop_first = True,
    dtype=int
)

print(X.head())


   Gender  Married  Dependents  Education  Self_Employed  ApplicantIncome  \
1       1        1           1          1              0             4583   
2       1        1           0          1              1             3000   
3       1        1           0          0              0             2583   
4       1        0           0          1              0             6000   
5       1        1           2          1              1             5417   

   CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
1             1508.0       128.0             360.0             1.0   
2                0.0        66.0             360.0             1.0   
3             2358.0       120.0             360.0             1.0   
4                0.0       141.0             360.0             1.0   
5             4196.0       267.0             360.0             1.0   

   Property_Area_Rural  Property_Area_Semiurban  Property_Area_Urban  
1                    1                       

In [99]:
print(X.isnull().sum())

Gender                     0
Married                    0
Dependents                 0
Education                  0
Self_Employed              0
ApplicantIncome            0
CoapplicantIncome          0
LoanAmount                 0
Loan_Amount_Term           0
Credit_History             0
Property_Area_Rural        0
Property_Area_Semiurban    0
Property_Area_Urban        0
dtype: int64


In [100]:
X = X.fillna(X.median())

In [101]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (384, 13)
Testing data: (96, 13)


In [102]:
scaler = StandardScaler()

In [103]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [104]:
model = LogisticRegression(max_iter = 1000)

In [105]:
model.fit(X_train , y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [106]:
y_pred = model.predict(X_test)

In [107]:
print("accuracy score = " , accuracy_score( y_test , y_pred))

accuracy score =  0.8333333333333334


In [108]:


print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.57      0.68        30
           1       0.83      0.95      0.89        66

    accuracy                           0.83        96
   macro avg       0.84      0.76      0.78        96
weighted avg       0.84      0.83      0.82        96

Confusion Matrix:
[[17 13]
 [ 3 63]]


In [109]:
knn_model = KNeighborsClassifier(n_neighbors=5)

In [110]:

knn_model.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [111]:
y_pred_knn = knn_model.predict(X_test)

In [112]:
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

KNN Accuracy: 0.8125


In [113]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_knn)
    ]
})

print(results)

                 Model  Accuracy
0  Logistic Regression  0.833333
1                  KNN  0.812500


In [115]:
import pickle

with open("loan_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("loan_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("loan_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("Model, scaler, and columns saved successfully!")

Model, scaler, and columns saved successfully!


In [116]:
print(X.columns.tolist())

['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area_Rural', 'Property_Area_Semiurban', 'Property_Area_Urban']


In [117]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.
